In [31]:
import sys
import pandas as pd
import numpy as np
import torch

from sklearn.metrics import mean_squared_error, mean_absolute_error
from transformers import Trainer, TrainingArguments
from transformers import PatchTSTConfig, PatchTSTForPrediction

sys.path.append('../src')

from dataset import normalize_time_series
from config import DF_VOL_FILE

# ==================== SETUP INICIAL ====================
# Detecta GPU disponível (mais rápido) ou usa CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Precision: bfloat16 é mais rápido em GPU modernas (Ampere+), float32 é universal
dtype = torch.bfloat16 if (device.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float32
print(f"Device: {device} | Precision: {dtype}")

# Seed para resultados reproduzíveis
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

dados = pd.read_csv(DF_VOL_FILE, parse_dates=['date'])

Device: cpu | Precision: torch.float32


In [32]:
# ==================== CONFIGURAÇÕES PRÉ-TREINAMENTO ====================
TIMESTAMP_COLUMN = 'date'  # Coluna com timestamps
TARGET_COLUMN = ['Vol']     # Coluna a prever (volatilidade)
ID_COLUMNS = []             # Sem IDs (série única)

# Tamanhos das janelas: histórico 7 dias, previsão 1 dia (dados diários)
CONTEXT_LENGTH = 512         # Janela histórica: x dias passados
FORECAST_HORIZON = 96        # Prever: x dias à frente

TRAIN_FRAC, VALID_FRAC = 0.7, 0.1  # Frações treino/validação/teste

# Hyperparâmetros do modelo
PATCH_LENGTH = 1            # Tamanho do patch (1=sem patchificação, mantém cada dia)
BATCH_SIZE = 32             # Samples por batch (reduzir se GPU memory limitada)
NUM_WORKERS = 4             # Workers para data loading (0 em Windows)
EPOCHS = 30                 # Reduzido: 50→30 (volatilidade tem ciclos curtos)
LEARNING_RATE = 1e-3        # Taxa de aprendizado

In [33]:
from tsfm_public.toolkit.util import select_by_index

dados.sort_values(by=TIMESTAMP_COLUMN, inplace=True)

n = len(dados)
train_end = int(n * TRAIN_FRAC)
valid_end = int(n * (TRAIN_FRAC + VALID_FRAC))

train_df = select_by_index(dados, start_index=None, end_index=train_end)
valid_df = select_by_index(dados, start_index=train_end - CONTEXT_LENGTH, end_index=valid_end)
test_df = select_by_index(dados, start_index=valid_end - CONTEXT_LENGTH, end_index=None)

In [34]:
# ==================== PRÉ-PROCESSAMENTO: NORMALIZAÇÃO ====================
from tsfm_public.toolkit.time_series_preprocessor import TimeSeriesPreprocessor
from tsfm_public.toolkit.dataset import ForecastDFDataset

# Normaliza dados: subtrai média, divide por std
# Essencial para modelos deep learning (converge melhor)
tsp = TimeSeriesPreprocessor(
    timestamp_column=TIMESTAMP_COLUMN,
    target_column=TARGET_COLUMN,
    id_columns=ID_COLUMNS,
    scaling='std'  # standardscaler: (x - mean)/std
)
tsp.train(train_df)  # Aprende mean/std no treino

def make_ds(df):
    """Cria dataset com janelas deslizantes"""
    return ForecastDFDataset(
        tsp.preprocess(df),  # Normaliza usando parâmetros do treino
        id_columns=ID_COLUMNS,
        target_columns=TARGET_COLUMN,
        context_length=CONTEXT_LENGTH,      # Quantos dias de histórico usar
        prediction_length=FORECAST_HORIZON, # Quantos dias prever
    )

X_train, X_valid, X_test = map(make_ds, [train_df, valid_df, test_df])
print(f"Treino: {len(X_train)} amostras | Val: {len(X_valid)} | Teste: {len(X_test)}")

Treino: 1183 amostras | Val: 161 | Teste: 417


In [59]:
# ==================== CARREGAR MODELO PRÉ-TREINADO ====================

BASE_MODEL = "ibm-granite/granite-timeseries-patchtst"

# Configuração: adapta modelo genérico para seu dataset
config = PatchTSTConfig.from_pretrained(BASE_MODEL)
config.num_input_channels = len(TARGET_COLUMN)  # 1 feature (Vol)

# Carrega weights pré-treinados
model = PatchTSTForPrediction.from_pretrained(
    BASE_MODEL,
    config=config
).to(device).to(dtype)

print(f"Modelo carregado: {BASE_MODEL}")
print(f"Num parâmetros: {model.num_parameters():,}")

Modelo carregado: ibm-granite/granite-timeseries-patchtst
Num parâmetros: 614,496


In [ ]:
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="metric_temp",
        label_names=["future_values"]
    )
)

# Predições
outputs = trainer.predict(X_test)

In [122]:
preds = outputs.predictions  # Formato: (num_samples, num_janelas, forecast_horizon)

len(preds), len(preds[0]), len(preds[0][0]), len(preds[0][0][0])

(3, 417, 96, 1)